In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np

# --- Configuration ---
# Task 1: Epoch별 mIoU 비교용 경로
PATH_EPOCH_BASE = '/home/jpsong/imgProcessing_Solo/runs_celebamaskhq/eval_report.txt'
PATH_EPOCH_TUNED = '/home/jpsong/imgProcessing_Solo/runs_celebamaskhq_hyperparam_tuning/eval_report.txt'

# Task 2: 특정 클래스(neck_l, ear_r) 성능 비교용 경로
PATH_FINAL_V1 = '/home/jpsong/imgProcessing_Solo/runs_celebamaskhq_hyperparam_tuning/eval_report_final.txt'
PATH_FINAL_V2 = '/home/jpsong/imgProcessing_Solo/runs_celebamaskhq_hyperparam_tuning_V2/eval_report_final.txt'

# Task 3: 전체 클래스 정렬 분석용 경로
PATH_FINAL_SINGLE = '/home/jpsong/imgProcessing_Solo/runs_celebamaskhq_hyperparam_tuning/eval_report_final.txt'

In [ ]:
def parse_epoch_history(file_path):
    """Epoch 및 mIoU 히스토리 파싱"""
    epochs, mious = [], []
    current_epoch = None
    
    if not os.path.exists(file_path):
        return epochs, mious

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith("Epoch:"):
                try:
                    current_epoch = int(line.split(":")[1].strip())
                except ValueError:
                    continue
            elif line.startswith("mIoU:") and current_epoch is not None:
                try:
                    val = float(line.split(":")[1].strip())
                    epochs.append(current_epoch)
                    mious.append(val)
                    current_epoch = None
                except ValueError:
                    continue
    return epochs, mious

def parse_final_report(file_path):
    """최종 결과 파일 파싱 (Dictionary 형태 반환)"""
    data = {}
    if not os.path.exists(file_path):
        return data

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            # Format: id(0) class(1) support(2) IoU(3) Dice(4) PA(5)
            if len(parts) >= 6:
                # Header나 구분선 제외 로직
                if not parts[0].isdigit(): 
                    continue
                
                cls_name = parts[1]
                try:
                    data[cls_name] = {
                        'support': int(parts[2]),
                        'iou': float(parts[3]),
                        'pa': float(parts[5])
                    }
                except ValueError:
                    continue
    return data